# UNIVERSITY OF EAST ANGLIA

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException, StaleElementReferenceException
import pandas as pd
import time

# Set up the Chrome webdriver (update with your chromedriver path)
driver = webdriver.Chrome()

# Open the website
driver.get("https://www.uea.ac.uk/search/courses/")

# Function to extract data from a single page
def extract_data_from_page():
    # Wait for course elements to be present on the page
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.XPATH, '//ol/li/div'))
    )
    
    # Find course elements on the page (re-find after navigation)
    course_elements = driver.find_elements(By.XPATH, '//ol/li/div')
    
    # Lists to store extracted details
    course_names = []
    course_categories = []
    course_departments = []
    course_start_dates = []
    course_end_dates = []
    course_links=[]

    # Loop through each course element and extract details
    for course_element in course_elements:
        try:
            course_name = course_element.find_element(By.XPATH, './/h4/a/span/span').text
        except NoSuchElementException:
            course_name = 'NA'

        try:
            course_category = driver.find_element(By.XPATH, "//span[@class='px-2 mb-2 mr-2 text-xs uppercase border border-black rounded-md font-brandon-medium' and @data-primarycategory='true']").text
        except NoSuchElementException:
            course_category = 'NA'

        try:
            course_department = driver.find_element(By.XPATH, "//span[@class='px-2 mb-2 mr-2 text-xs uppercase border border-black rounded-md font-brandon-medium' and @data-department='true']").text
        except NoSuchElementException:
            course_department = 'NA'

        try:
            course_start_date = course_element.find_element(By.XPATH, './/div/span[@class="px-2 mb-2 mr-2 text-xs uppercase border border-black rounded-md font-brandon-medium flex flex-grid"]').text
        except NoSuchElementException:
            course_start_date = 'NA'

        try:
            course_end_date = course_element.find_element(By.XPATH, './/div/span[@class="px-2 mb-2 mr-2 text-xs uppercase border border-black rounded-md font-brandon-medium flex flex-grid"]').text
        except NoSuchElementException:
            course_end_date = 'NA'
            
        try:
            course_link_element = course_element.find_element(By.TAG_NAME, "a")
            course_link = course_link_element.get_attribute("href")
        except:
            course_link = "NA"
        


        # Append extracted details to respective lists
        course_names.append(course_name)
        course_categories.append(course_category)
        course_departments.append(course_department)
        course_start_dates.append(course_start_date)
        course_end_dates.append(course_end_date)
        course_links.append(course_link)

    # Construct a DataFrame
    data = {
        'Course_Title': course_names,
        'Degree_Level': course_categories,
        'Course_Domain': course_departments,
        'Start_Date': course_start_dates,
        'End_Date': course_end_dates,
        'Course_Link':course_links
    }

    df = pd.DataFrame(data)
    return df

# Get the total number of pages
pagination_elements = driver.find_elements(By.XPATH, '//ul[@class="flex justify-center my-6"]/li/a')
total_pages = 70

# Initialize an empty DataFrame to store all data
all_data = pd.DataFrame()

# Iterate through all pages
for page_number in range(total_pages):
    # Extract data from the current page
    try:
        extracted_data = extract_data_from_page()
    except StaleElementReferenceException:
        print("Stale Element Reference Exception. Retrying...")
        extracted_data = extract_data_from_page()  # Retry extraction
    
    # Concatenate the extracted data to the existing data
    all_data = pd.concat([all_data, extracted_data], ignore_index=True)

    # Click on the next page link using JavaScript
    try:
        next_page_link = driver.find_element(By.XPATH, f'//ul[@class="flex justify-center my-6"]/li/a[text()="{page_number + 2}"]')
        driver.execute_script("arguments[0].click();", next_page_link)
        time.sleep(5)  # Add a delay to allow the page to load (adjust as needed)
    except (ElementClickInterceptedException, NoSuchElementException):
        print("Element click intercepted or element not found. Trying with JavaScript.")
        driver.execute_script("arguments[0].click();", next_page_link)
        time.sleep(5)  # Add a delay to allow the page to load (adjust as needed)

# Close the webdriver
driver.quit()

# Print the combined dataframe
all_data


Element click intercepted or element not found. Trying with JavaScript.
Element click intercepted or element not found. Trying with JavaScript.
Element click intercepted or element not found. Trying with JavaScript.
Element click intercepted or element not found. Trying with JavaScript.
Element click intercepted or element not found. Trying with JavaScript.


,Course_Title,Degree_Level,Course_Domain,Start_Date,End_Date,Course_Link
0,MSc Economics,POSTGRADUATE,ECONOMICS,SEPTEMBER 2023,SEPTEMBER 2023,https://www.uea.ac.uk/course/postgraduate/msc-...
1,BSc (Hons) Economics,POSTGRADUATE,ECONOMICS,SEPTEMBER 2023,SEPTEMBER 2023,https://www.uea.ac.uk/course/undergraduate/bsc...
2,MSc Business Management,POSTGRADUATE,ECONOMICS,SEPTEMBER 2023,SEPTEMBER 2023,https://www.uea.ac.uk/course/postgraduate/msc-...
3,MBBS Medicine,POSTGRADUATE,ECONOMICS,SEPTEMBER 2023,SEPTEMBER 2023,https://www.uea.ac.uk/course/undergraduate/mbb...
4,MSc Computing Science,POSTGRADUATE,ECONOMICS,SEPTEMBER 2023,SEPTEMBER 2023,https://www.uea.ac.uk/course/postgraduate/msc-...
...,...,...,...,...,...,...
825,Clinical Associate in Psychology Masters Appre...,INTO,SOCIAL WORK,SEPTEMBER 2024,SEPTEMBER 2024,https://www.uea.ac.uk/course/apprenticeship/ma...
826,Healthcare Science Practitioner (Applied Biome...,INTO,SOCIAL WORK,SEPTEMBER 2024,SEPTEMBER 2024,https://www.uea.ac.uk/course/apprenticeship/he...
827,Certificate of Higher Education in Pharmacy Te...,INTO,SOCIAL WORK,SEPTEMBER 2023,SEPTEMBER 2023,https://www.uea.ac.uk/course/apprenticeship/ce...
828,Advanced Clinical Practitioner Masters Degree ...,INTO,SOCIAL WORK,SEPTEMBER 2024,SEPTEMBER 2024,https://www.uea.ac.uk/course/apprenticeship/ad...


In [4]:
# Save the data to a CSV file
all_data.to_csv("UniversityOfEastAngelia.csv", index=False)